# 09 回测引擎：手写最小口径与 `bt` 复现

## 9.1 本章目标

第 08 章已经把因子分数转成目标权重。本章回答一个更关键的问题：如果按这些目标权重交易，净值曲线应该怎样计算？

本章会讲清楚四个口径：

- 资产收益率来自 `close.pct_change()`。
- 当天目标权重不能用于当天收益，持仓要用 `target_weights.shift(1)`。
- 交易成本用换手率近似：`cost = turnover * cost_bps / 10000`。
- 净值曲线为 `NAV = (1 + strategy_return).cumprod()`。

## 9.2 前置条件

- 已理解 pandas 的矩阵运算。
- 已运行或理解第 06-08 章的因子、打分和 TopN 权重。

## 9.3 输入与输出

输入是价格矩阵和目标权重矩阵。输出是策略收益、成本、换手和净值，并保存：

- `outputs/results/chapter09_strategy_returns.csv`
- `outputs/results/chapter09_target_weights.csv`

## 9.4 本章位置

`目标权重 -> 滞后持仓 -> 策略收益/NAV -> 第 10 章绩效评估`

## 9.5 学习路线

1. 先手写最小回测函数。
2. 看清 `shift(1)` 为什么重要。
3. 用第 08 章同一套 ETF 权重跑实战数据。
4. 对比 `lib.backtest` 与开源库 `bt`。
5. 把结果交给第 10 章评估。


In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "lib").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Cannot find pyquant-roadmap project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

COST_BPS = 8.0

from lib.paths import RESULTS_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pd.Series({"project_root": ".", "results_dir": RESULTS_DIR.relative_to(PROJECT_ROOT).as_posix()}, name="value")


project_root                  .
results_dir     outputs/results
Name: value, dtype: object

## 9.6 手写最小数据

最小回测只需要四类表：

- `price`：资产收盘价矩阵。
- `asset_returns`：资产收益率矩阵。
- `target_weights`：策略希望持有的目标权重。
- `positions`：实际用于计算收益的持仓，也就是 `target_weights.shift(1)`。

下面用 5 个交易日、2 只 ETF 做最小例子。


In [2]:
tiny_dates = pd.bdate_range("2024-01-02", periods=5)

tiny_price = pd.DataFrame(
    {
        "ETF_A": [100.0, 102.0, 101.0, 103.0, 104.0],
        "ETF_B": [100.0, 99.0, 101.0, 100.0, 102.0],
    },
    index=tiny_dates,
)

tiny_target_weights = pd.DataFrame(
    {
        "ETF_A": [1.0, 1.0, 0.5, 0.0, 0.0],
        "ETF_B": [0.0, 0.0, 0.5, 1.0, 1.0],
    },
    index=tiny_dates,
)

tiny_asset_returns = tiny_price.pct_change().fillna(0.0)

display(tiny_price)
display(tiny_target_weights)


,ETF_A,ETF_B
2024-01-02,100.0,100.0
2024-01-03,102.0,99.0
2024-01-04,101.0,101.0
2024-01-05,103.0,100.0
2024-01-08,104.0,102.0


,ETF_A,ETF_B
2024-01-02,1.0,0.0
2024-01-03,1.0,0.0
2024-01-04,0.5,0.5
2024-01-05,0.0,1.0
2024-01-08,0.0,1.0


## 9.7 手写最小回测函数

回测函数的核心只有三步：先滞后目标权重，再用持仓乘以资产收益，最后扣除由换手率估算的交易成本。

关键公式是：

```python
positions = target_weights.shift(1).fillna(0.0)
gross = (positions * asset_returns).sum(axis=1)
```

这样可以避免“今天收盘后才知道的目标权重，却拿来赚今天收益”的未来函数。


In [3]:
def minimal_backtest(
    asset_returns: pd.DataFrame,
    target_weights: pd.DataFrame,
    cost_bps: float = COST_BPS,
) -> pd.DataFrame:
    aligned_weights = target_weights.reindex(index=asset_returns.index, columns=asset_returns.columns).fillna(0.0)

    positions = aligned_weights.shift(1).fillna(0.0)
    gross = (positions * asset_returns).sum(axis=1)

    turnover = aligned_weights.diff().abs().sum(axis=1)
    if not turnover.empty:
        turnover.iloc[0] = aligned_weights.iloc[0].abs().sum()
    turnover = turnover.fillna(0.0)

    cost = turnover * cost_bps / 10000.0
    strategy_return = gross - cost
    nav = (1.0 + strategy_return).cumprod()

    return pd.DataFrame(
        {
            "gross_return": gross,
            "cost": cost,
            "strategy_return": strategy_return,
            "nav": nav,
            "turnover": turnover,
        }
    )


tiny_report = minimal_backtest(tiny_asset_returns, tiny_target_weights, cost_bps=COST_BPS)
tiny_positions = tiny_target_weights.shift(1).fillna(0.0)

display(tiny_positions.rename_axis(index="date"))
display(tiny_report)


,ETF_A,ETF_B
date,,
2024-01-02,0.0,0.0
2024-01-03,1.0,0.0
2024-01-04,1.0,0.0
2024-01-05,0.5,0.5
2024-01-08,0.0,1.0


,gross_return,cost,strategy_return,nav,turnover
2024-01-02,0.000000,0.0008,-0.000800,0.999200,1.0
2024-01-03,0.020000,0.0000,0.020000,1.019184,0.0
2024-01-04,-0.009804,0.0008,-0.010604,1.008377,1.0
2024-01-05,0.004950,0.0008,0.004150,1.012562,1.0
2024-01-08,0.020000,0.0000,0.020000,1.032813,0.0


### 9.7.1 如何读这个小账本

第一天虽然已经给出 `ETF_A` 的目标权重，但实际持仓还没有建立，所以 `gross_return` 为 0。

权重从 `ETF_A=1.0, ETF_B=0.0` 调到 `ETF_A=0.5, ETF_B=0.5` 时，卖出 0.5、买入 0.5，总换手为 1.0。`8 bps` 成本就是 `0.0008`。


In [4]:
manual_check = pd.concat(
    {
        "target_A": tiny_target_weights["ETF_A"],
        "position_A_used_for_return": tiny_positions["ETF_A"],
        "asset_return_A": tiny_asset_returns["ETF_A"],
        "turnover": tiny_report["turnover"],
        "cost": tiny_report["cost"],
        "strategy_return": tiny_report["strategy_return"],
        "nav": tiny_report["nav"],
    },
    axis=1,
)
manual_check


,target_A,position_A_used_for_return,asset_return_A,turnover,cost,strategy_return,nav
2024-01-02,1.0,0.0,0.000000,1.0,0.0008,-0.000800,0.999200
2024-01-03,1.0,1.0,0.020000,0.0,0.0000,0.020000,1.019184
2024-01-04,0.5,1.0,-0.009804,1.0,0.0008,-0.010604,1.008377
2024-01-05,0.0,0.5,0.019802,1.0,0.0008,0.004150,1.012562
2024-01-08,0.0,0.0,0.009709,0.0,0.0000,0.020000,1.032813


## 9.8 对比错误口径：不滞后持仓

如果直接用当天目标权重乘以当天收益，就等于提前知道当天收盘后的决策。这种错误通常会把回测结果变得过于乐观。

下面只做对比，不把它作为实战口径。


In [5]:
lookahead_return = (tiny_target_weights * tiny_asset_returns).sum(axis=1) - tiny_report["cost"]
lookahead_nav = (1.0 + lookahead_return).cumprod()

pd.DataFrame(
    {
        "correct_return_shifted": tiny_report["strategy_return"],
        "wrong_return_no_shift": lookahead_return,
        "correct_nav_shifted": tiny_report["nav"],
        "wrong_nav_no_shift": lookahead_nav,
    }
)


,correct_return_shifted,wrong_return_no_shift,correct_nav_shifted,wrong_nav_no_shift
2024-01-02,-0.000800,-0.000800,0.999200,0.999200
2024-01-03,0.020000,0.020000,1.019184,1.019184
2024-01-04,-0.010604,0.004399,1.008377,1.023667
2024-01-05,0.004150,-0.010701,1.012562,1.012713
2024-01-08,0.020000,0.020000,1.032813,1.032967


## 9.9 小练习：提高交易成本

把成本从 `8 bps` 提高到 `30 bps`，观察净值差距如何扩大。

这个练习提醒你：高换手策略对交易成本更敏感。


In [6]:
higher_cost_bps = 30.0

tiny_cost_compare = pd.DataFrame(
    {
        "nav_8bps": minimal_backtest(tiny_asset_returns, tiny_target_weights, cost_bps=8.0)["nav"],
        "nav_30bps": minimal_backtest(tiny_asset_returns, tiny_target_weights, cost_bps=higher_cost_bps)["nav"],
    }
)
tiny_cost_compare["gap"] = tiny_cost_compare["nav_8bps"] - tiny_cost_compare["nav_30bps"]
tiny_cost_compare


,nav_8bps,nav_30bps,gap
2024-01-02,0.999200,0.997000,0.002200
2024-01-03,1.019184,1.016940,0.002244
2024-01-04,1.008377,1.003919,0.004457
2024-01-05,1.012562,1.005877,0.006685
2024-01-08,1.032813,1.025995,0.006818


## 9.10 回到第 08 章的 ETF 权重

现在把同一套回测口径用到真实缓存 ETF 数据：

1. 读取缓存 ETF 价格。
2. 构建技术因子。
3. 合成多因子 TopN 分数。
4. 生成目标权重矩阵。
5. 计算收益、成本、换手和净值。

这一步把第 08 章的组合构建接到回测环节。


In [7]:
from lib.data import load_sample_prices
from lib.factors import build_technical_factor_panel, combine_score, zscore_by_date
from lib.portfolio import top_n_equal_weight, weights_to_matrix


def month_end_trading_dates(dates: pd.Series) -> pd.Series:
    unique_dates = pd.Series(pd.to_datetime(dates.dropna().unique())).sort_values()
    return unique_dates.groupby(unique_dates.dt.to_period("M")).max()


prices = load_sample_prices()
prices["date"] = pd.to_datetime(prices["date"])
prices["code"] = prices["code"].astype(str)

close = prices.pivot(index="date", columns="code", values="close").sort_index().astype(float)
asset_returns = close.pct_change().fillna(0.0)

factor_cols = ["momentum_60", "low_vol_20", "ma_gap_20_60"]
factor_weights = {"momentum_60": 0.45, "low_vol_20": 0.35, "ma_gap_20_60": 0.20}

factors = build_technical_factor_panel(prices)
scored = combine_score(zscore_by_date(factors, factor_cols), factor_weights)
rebalance_dates = month_end_trading_dates(scored["date"])
rebalance_scores = scored[scored["date"].isin(rebalance_dates)]

sparse_weights = top_n_equal_weight(rebalance_scores, n=3)
target_weights = weights_to_matrix(sparse_weights, close.index, close.columns, carry_forward=True)

summary = pd.Series(
    {
        "price_rows": len(prices),
        "asset_count": close.shape[1],
        "first_price_date": close.index.min().date(),
        "last_price_date": close.index.max().date(),
        "rebalance_count": len(rebalance_dates),
        "first_rebalance_date": rebalance_dates.min().date(),
        "last_rebalance_date": rebalance_dates.max().date(),
    }
)
summary


price_rows                    2900
asset_count                      4
first_price_date        2021-01-04
last_price_date         2023-12-29
rebalance_count                 33
first_rebalance_date    2021-04-30
last_rebalance_date     2023-12-29
dtype: object

In [8]:
display(sparse_weights.tail(9))
display(target_weights.tail())


,date,code,weight
90,2023-10-31,510300,0.333333
91,2023-10-31,510500,0.333333
92,2023-10-31,512100,0.333333
93,2023-11-30,510300,0.333333
94,2023-11-30,510500,0.333333
95,2023-11-30,512100,0.333333
96,2023-12-29,510300,0.333333
97,2023-12-29,510500,0.333333
98,2023-12-29,512100,0.333333


code,159915,510300,510500,512100
date,,,,
2023-12-25,0.0,0.333333,0.333333,0.333333
2023-12-26,0.0,0.333333,0.333333,0.333333
2023-12-27,0.0,0.333333,0.333333,0.333333
2023-12-28,0.0,0.333333,0.333333,0.333333
2023-12-29,0.0,0.333333,0.333333,0.333333


## 9.11 用目标权重计算 ETF 回测

`target_weights` 代表调仓日形成的目标持仓。回测收益使用 `t-1` 的持仓赚取 `t` 日收益。


In [9]:
manual_report = minimal_backtest(asset_returns, target_weights, cost_bps=COST_BPS)

report_preview = manual_report.tail().copy()
report_preview["nav"] = report_preview["nav"].round(4)
report_preview


,gross_return,cost,strategy_return,nav,turnover
date,,,,,
2023-12-25,-0.000088,0.0,-0.000088,0.7434,0.0
2023-12-26,-0.009641,0.0,-0.009641,0.7363,0.0
2023-12-27,0.004080,0.0,0.004080,0.7393,0.0
2023-12-28,0.022723,0.0,0.022723,0.7561,0.0
2023-12-29,0.008851,0.0,0.008851,0.7628,0.0


In [10]:
quality_checks = pd.Series(
    {
        "weights_never_above_100pct": bool((target_weights.sum(axis=1) <= 1.0 + 1e-9).all()),
        "turnover_nonnegative": bool((manual_report["turnover"] >= -1e-12).all()),
        "nav_always_positive": bool((manual_report["nav"] > 0).all()),
        "first_day_no_position_return": bool(abs(manual_report["gross_return"].iloc[0]) < 1e-12),
    }
)
display(quality_checks)
assert quality_checks.all()


weights_never_above_100pct      True
turnover_nonnegative            True
nav_always_positive             True
first_day_no_position_return    True
dtype: bool

## 9.12 `bt` 框架：不是 Backtrader 的 Cerebro

你可能在其他教程里见过 `Cerebro`。`Cerebro` 是 **Backtrader** 的核心对象，不是本项目使用的框架。

本项目选择 **`bt`**，因为当前主线是权重型、低频组合回测，`bt` 对“目标权重矩阵 -> 再平衡 -> 回测结果”的表达更直接。

`bt` 的核心对象包括：

- `bt.Strategy`：定义策略名称和算法链。
- `bt.algos.RunDaily()`：每天检查是否运行。
- `bt.algos.SelectAll()`：选择所有资产。
- `bt.algos.WeighTarget(weights)`：读取目标权重矩阵。
- `bt.algos.Rebalance()`：按目标权重再平衡。
- `bt.Backtest(strategy, price)`：把策略和价格数据绑定。
- `bt.run(backtest)`：返回 `Result` 对象。

`bt` 负责策略调度和回测框架；本项目仍保留 `returns_from_weights`，用于透明地展示收益、成本和换手的计算口径。


In [11]:
import bt

bt_components = pd.DataFrame(
    [
        ("bt.Strategy", "把一组 algo 组合成策略"),
        ("bt.algos.RunDaily", "每天检查是否运行策略"),
        ("bt.algos.SelectAll", "选择价格表中的所有资产"),
        ("bt.algos.WeighTarget", "读取目标权重矩阵"),
        ("bt.algos.Rebalance", "按目标权重再平衡"),
        ("bt.Backtest", "绑定策略和价格数据"),
        ("bt.run(...) -> Result", "运行回测并返回结果对象"),
    ],
    columns=["bt object", "role"],
)

print("bt version:", getattr(bt, "__version__", "unknown"))
bt_components


bt version: 1.2.0


,bt object,role
0,bt.Strategy,把一组 algo 组合成策略
1,bt.algos.RunDaily,每天检查是否运行策略
2,bt.algos.SelectAll,选择价格表中的所有资产
3,bt.algos.WeighTarget,读取目标权重矩阵
4,bt.algos.Rebalance,按目标权重再平衡
5,bt.Backtest,绑定策略和价格数据
6,bt.run(...) -> Result,运行回测并返回结果对象


In [12]:
bt_strategy = bt.Strategy(
    "chapter09_direct_bt",
    [
        bt.algos.RunDaily(),
        bt.algos.SelectAll(),
        bt.algos.WeighTarget(target_weights),
        bt.algos.Rebalance(),
    ],
)
bt_backtest = bt.Backtest(bt_strategy, close)
bt_result_direct = bt.run(bt_backtest)

print(type(bt_result_direct).__name__)
print("backtest_count:", len(bt_result_direct.backtests))


Result
backtest_count: 1


## 9.13 用 `lib.backtest` 复现实战口径

实际 notebook 不应该每章重复写 `bt.Strategy(...)`。所以 `lib.backtest` 提供两个入口：

- `returns_from_weights(price, target_weights, cost_bps)`：输出 `gross_return / cost / strategy_return / nav / turnover`。
- `run_bt_from_weights(price, target_weights, name)`：复用 `bt` 跑出框架结果。

第 10 章会继续使用 `returns_from_weights` 的透明结果做绩效评估，`bt` 结果用于展示开源框架复现方式。


In [13]:
from lib.backtest import returns_from_weights, run_bt_from_weights

practical_report = returns_from_weights(close, target_weights, cost_bps=COST_BPS)
bt_result = run_bt_from_weights(close, target_weights, name="chapter09_etf_strategy")

comparison = pd.DataFrame(
    {
        "manual_nav": manual_report["nav"],
        "lib_nav": practical_report["nav"],
        "manual_return": manual_report["strategy_return"],
        "lib_return": practical_report["strategy_return"],
    }
)
max_abs_nav_diff = float((comparison["manual_nav"] - comparison["lib_nav"]).abs().max())
max_abs_return_diff = float((comparison["manual_return"] - comparison["lib_return"]).abs().max())

print("bt result type:", type(bt_result).__name__)
print("bt backtest count:", len(bt_result.backtests))
print("max_abs_nav_diff:", max_abs_nav_diff)
print("max_abs_return_diff:", max_abs_return_diff)
assert max_abs_nav_diff < 1e-12
assert max_abs_return_diff < 1e-12

comparison.tail()


bt result type: Result
bt backtest count: 1
max_abs_nav_diff: 0.0
max_abs_return_diff: 0.0


,manual_nav,lib_nav,manual_return,lib_return
date,,,,
2023-12-25,0.743436,0.743436,-0.000088,-0.000088
2023-12-26,0.736269,0.736269,-0.009641,-0.009641
2023-12-27,0.739273,0.739273,0.004080,0.004080
2023-12-28,0.756072,0.756072,0.022723,0.022723
2023-12-29,0.762763,0.762763,0.008851,0.008851


## 9.14 保存结果给第 10 章

第 10 章需要策略收益和净值曲线。这里保存本章回测结果，方便后续评估、画图和报告输出。


In [14]:
strategy_returns_path = RESULTS_DIR / "chapter09_strategy_returns.csv"
target_weights_path = RESULTS_DIR / "chapter09_target_weights.csv"

practical_report.to_csv(strategy_returns_path, index=True, index_label="date", encoding="utf-8-sig")
target_weights.to_csv(target_weights_path, index=True, index_label="date", encoding="utf-8-sig")

saved_outputs = pd.Series(
    {
        "strategy_returns": str(strategy_returns_path.relative_to(PROJECT_ROOT)),
        "target_weights": str(target_weights_path.relative_to(PROJECT_ROOT)),
        "final_nav": round(float(practical_report["nav"].iloc[-1]), 4),
        "average_daily_turnover": round(float(practical_report["turnover"].mean()), 6),
        "total_cost": round(float(practical_report["cost"].sum()), 6),
    }
)
saved_outputs


strategy_returns          outputs\results\chapter09_strategy_returns.csv
target_weights              outputs\results\chapter09_target_weights.csv
final_nav                                                         0.7628
average_daily_turnover                                          0.013333
total_cost                                                      0.007733
dtype: object

## 9.15 本章小结

你已经完成回测引擎的核心口径：

- 用资产收益和滞后持仓计算策略收益。
- 用换手率近似交易成本。
- 用累计收益生成净值曲线。
- 用 `bt` 复现权重型回测框架。
- 用 `returns_from_weights` 保留透明、可检查的收益序列。

## 9.16 交接给第 10 章

第 10 章会把 `strategy_return` 和 `nav` 转成更可读的结果：

- 绩效指标。
- 回撤图。
- 策略报告和 HTML 报告。
